In [1]:
from project_paths import paths
import polars as pl
from imd_features.config import FeatureSetConfig
from imd_features.evaluate import evaluate

output_dir = paths.output

In [2]:
config_files = output_dir.glob("*_config.json")

all_results = {}
for config_path in config_files:
    config = FeatureSetConfig.model_validate_json(config_path.read_text())
    df = pl.read_parquet(output_dir / f"{config.output_name}.parquet")
    all_results[config.output_name] = evaluate(df, config)

c:\Users\Daniel\projects\imd_feature_engineering\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.2314211932024065e-17.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
c:\Users\Daniel\projects\imd_feature_engineering\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.2867759236014729e-17.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
c:\Users\Daniel\projects\imd_feature_engineering\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.434554952494568e-17.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
c:\Users\Daniel\projects\imd_feature_engineering\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.202

In [3]:
leaderboard_rows = []
for name, results in all_results.items():
    for model_name, r in results.items():
        leaderboard_rows.append({
            "feature_set": name,
            "model": model_name,
            "r2_mean": r["r2_mean"],
            "rmse_mean": r["rmse_mean"],
            "spearman_mean": r["spearman_mean"],
        })

leaderboard = pl.DataFrame(leaderboard_rows).sort("r2_mean", descending=True)

with pl.Config(tbl_rows=150):
    print(leaderboard)

shape: (42, 5)
┌─────────────────────────────────┬───────────────┬───────────┬───────────┬───────────────┐
│ feature_set                     ┆ model         ┆ r2_mean   ┆ rmse_mean ┆ spearman_mean │
│ ---                             ┆ ---           ┆ ---       ┆ ---       ┆ ---           │
│ str                             ┆ str           ┆ f64       ┆ f64       ┆ f64           │
╞═════════════════════════════════╪═══════════════╪═══════════╪═══════════╪═══════════════╡
│ engineered_rates_8821352e       ┆ random_forest ┆ 0.87509   ┆ 5.534152  ┆ 0.93219       │
│ engineered_rates_v2_c3ad0464    ┆ random_forest ┆ 0.872597  ┆ 5.574342  ┆ 0.936976      │
│ compact_30_3b29b236             ┆ random_forest ┆ 0.832163  ┆ 6.484381  ┆ 0.918187      │
│ calibrated_reduction_a5381280   ┆ random_forest ┆ 0.827699  ┆ 6.555838  ┆ 0.914199      │
│ all_features_unreduced_dab4535… ┆ random_forest ┆ 0.822766  ┆ 6.651824  ┆ 0.90802       │
│ all_features_unreduced_e2827a3… ┆ random_forest ┆ 0.821074  ┆ 6